# Contrail-detector filter playground

Interactive explorer for the detector pipeline. Stack up to three
preprocessing transforms, tune Canny + Hough, and inspect every labelled
candidate as TP / FN / FP / TN at the current Youden-J threshold.

## Data scope

- **Candidate pool:** every episode covered by the unified prash + thendo +
  reviewer-1 label set on 2026-04-09 (≈128 episodes, 53 contrail / 75
  no_contrail). Loaded from the public-bundle `manifest.json` +
  `projections.jsonl` so the playground sees the same data the HPO sweep
  saw.
- **April-8 sentinels:** the 7 hand-picked positives from batch-1 stay as a
  regression check.

## What this notebook shows

Four panels per candidate:
1. **Crop + rotated polygon** — the frame the pipeline hands to `detect()`.
2. **Chain output (gray)** — what `apply_chain` produces, using the current
   transform chain and per-transform parameter sliders.
3. **Canny edges** — edges inside the rotated polygon mask, using the
   post-chain gray image.
4. **Hough overlay** — aligned long lines in green, rejected lines in
   orange, picked `pixel_line` in thick bright-green.

## Temporal-diff lookback experiment

`temporal_diff` was added to the chain to suppress static structure (e.g.
the building edge in the camera FOV). The lookback knob controls how many
seconds back the prev-frame comes from:

- **1 s** (default, matches the live pipeline): kills moving cloud edges
  but jitter / sun-shadow drift can leak through on a high-contrast
  building.
- **3–10 s**: more aggressive cancellation of slow drift; aircraft has
  moved further so the contrail starts to ghost in the diff.

Pick a candidate, set chain A to `temporal_diff`, chain B to `cross_grad`,
and sweep the lookback knob to see whether longer lookback removes
building FPs without losing recall on real positives.

## Setup

```
uv sync --extra review
uv run jupyter lab notebooks/filter_playground.ipynb
```

In [1]:
from __future__ import annotations

import datetime
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import (
    Button, Checkbox, Dropdown, FloatSlider, HBox, IntSlider, Label, Layout,
    Output, VBox, interactive_output,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from concam.config import DetectionConfig, load_config
from concam.detection import detect
from concam.detection.transforms import (
    DISPLAY_COLORMAPS, DISPLAY_LABELS, TRANSFORMS, TRANSFORMS_BY_NAME, apply_chain,
)
from concam.projection import PixelPoint, Rect, rotated_polygon

SITE_CONFIG = load_config(REPO_ROOT / "configs" / "mit_green_building.yaml")
DET_CFG = SITE_CONFIG.detection
CALIB_W, CALIB_H = (
    int(SITE_CONFIG.calibration.calibration_resolution[0]),
    int(SITE_CONFIG.calibration.calibration_resolution[1]),
)
EXTRACT_PAD = 20

# --- April-9 data sources (manifest + projections + 3-reviewer labels) ---
APRIL9_DATE = "2026-04-09"
APRIL9_MANIFEST_PATH = Path.home() / "public_html" / "concam" / APRIL9_DATE / "manifest.json"
APRIL9_PROJECTIONS_PATH = REPO_ROOT / "output" / APRIL9_DATE / "projections.jsonl"
APRIL9_VIDEO_PATH = Path(f"/net/d16/data/contrail-camera/{APRIL9_DATE.replace('-', '_')}_0000_2359.mp4")
APRIL9_LABEL_PATHS = [
    REPO_ROOT / "labels" / f"{APRIL9_DATE}_prash.json",
    REPO_ROOT / "labels" / f"{APRIL9_DATE}_thendo.json",
    REPO_ROOT / "labels" / f"{APRIL9_DATE}_reviewer-1.json",
]

# Lookbacks (in seconds) that the prev-frame slider can pick. Frame 1 is
# pre-decoded at load time (current pipeline default); 3/5/10 are decoded
# lazily on first access via _ensure_prev_for_lookback().
PREV_LOOKBACKS_S = (1, 3, 5, 10)
DEFAULT_LOOKBACK_S = 1

# --- April-8 sentinels (unchanged from previous playground) ---
APRIL8_DIR = REPO_ROOT / "output" / "validation" / "detection" / "2026-04-08"
APRIL8_MANIFEST_PATH = APRIL8_DIR / "manifest.json"
APRIL8_LABELS_PATH = APRIL8_DIR / "labels.json"

print(f"April-9 manifest:    {APRIL9_MANIFEST_PATH}")
print(f"April-9 projections: {APRIL9_PROJECTIONS_PATH}")
print(f"April-9 video:       {APRIL9_VIDEO_PATH}")
print(f"April-9 labels:      {len(APRIL9_LABEL_PATHS)} reviewer files")
print(f"April-8 manifest:    {APRIL8_MANIFEST_PATH}")

April-9 manifest:    /home/prash/public_html/concam/2026-04-09/manifest.json
April-9 projections: /home/prash/contrails/mit-concam-pipeline/output/2026-04-09/projections.jsonl
April-9 video:       /net/d16/data/contrail-camera/2026_04_09_0000_2359.mp4
April-9 labels:      3 reviewer files
April-8 manifest:    /home/prash/contrails/mit-concam-pipeline/output/validation/detection/2026-04-08/manifest.json


## Load candidates

**April-9** candidates are built from the public-bundle `manifest.json`
joined with `projections.jsonl`, restricted to episodes covered by at
least one of `labels/2026-04-09_{prash,thendo,reviewer-1}.json`. Conflicts
between reviewers use the same first-file-wins rule as
`scripts/detection_hpo.py`.

For each labelled episode we pick its *peak* frame (highest detector score
across the episode; ties or all-zeros fall back to the middle frame), then
look up the projection at that wall_time_utc to get pixel_x/pixel_y +
path_dx/path_dy. The current frame and the prev frame at lookback 1 s are
decoded in a single sequential pass; longer lookbacks (3 / 5 / 10 s) are
decoded lazily the first time the slider lands on them.

**April-8** positive sentinels come from the existing batch-1
`manifest.json` + `labels.json`. Crops are read from the pre-rendered
`rois/roi_NN.png` files on disk. These positives have no multi-lookback
prev frames available, so the temporal-diff slider falls back to no diff
on them.

In [2]:
def _context_crop(frame, pixel_x, pixel_y, pad_px):
    h, w = frame.shape[:2]
    cx, cy = int(round(pixel_x)), int(round(pixel_y))
    x1 = max(0, cx - pad_px)
    y1 = max(0, cy - pad_px)
    x2 = min(w, cx + pad_px)
    y2 = min(h, cy + pad_px)
    return frame[y1:y2, x1:x2].copy(), (x1, y1)


# CROP_PAD_PX sets the half-width of the extracted crop in pixels.  Big
# enough that the largest slider value (roi_along_px=600) plus some margin
# fits inside; the rotated-polygon mask still constrains Canny to the
# along-track strip, so using a wider crop costs nothing.
CROP_PAD_PX = 400


def _decode_frames_sequential(video_path: Path, target_indices: set[int]) -> dict[int, np.ndarray]:
    """Single sequential video pass; return {frame_idx: bgr_ndarray} for matches."""
    if not target_indices:
        return {}
    out: dict[int, np.ndarray] = {}
    targets = sorted(target_indices)
    container = av.open(str(video_path))
    try:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        avg_rate = float(stream.average_rate or 1)
        time_base = float(stream.time_base or 1)
        first = targets[0]
        if avg_rate > 0 and time_base > 0:
            pts = int(first / avg_rate / time_base)
            container.seek(max(pts - 10, 0), stream=stream, any_frame=False)
        wi = 0
        for pkt in container.decode(stream):
            if pkt.pts is None:
                continue
            fidx = int(round(float(pkt.pts) * time_base * avg_rate))
            while wi < len(targets) and targets[wi] < fidx:
                wi += 1
            if wi >= len(targets):
                break
            if fidx == targets[wi]:
                out[fidx] = pkt.to_ndarray(format="bgr24")
                wi += 1
    finally:
        container.close()
    return out


def _decode_one_frame(video_path: Path, target_idx: int) -> np.ndarray | None:
    """Seek + decode a single frame. Used for lazy multi-lookback fills."""
    if target_idx < 0:
        return None
    container = av.open(str(video_path))
    try:
        stream = container.streams.video[0]
        avg_rate = float(stream.average_rate or 1)
        time_base = float(stream.time_base or 1)
        if avg_rate <= 0 or time_base <= 0:
            return None
        pts = int(target_idx / avg_rate / time_base)
        container.seek(max(pts - 10, 0), stream=stream, any_frame=False)
        for pkt in container.decode(stream):
            if pkt.pts is None:
                continue
            fidx = int(round(float(pkt.pts) * time_base * avg_rate))
            if fidx >= target_idx:
                return pkt.to_ndarray(format="bgr24")
    finally:
        container.close()
    return None


# --- Merge labels (matches detection_hpo.merge_labels: first-file-wins) ---
def _load_label_dict(path: Path) -> dict[int, str]:
    data = json.loads(path.read_text())
    out: dict[int, str] = {}
    for entry in data["labels"]:
        lbl = entry.get("label")
        if lbl in ("contrail", "no_contrail"):  # match scripts/detection_hpo.py: drop unsure at load
            out[int(entry["episode_id"])] = lbl
    return out


def _merge_labels(paths: list[Path]) -> dict[int, str]:
    unified: dict[int, str] = {}
    for p in paths:
        if not p.exists():
            print(f"  WARN: label file missing: {p}")
            continue
        labels = _load_label_dict(p)
        labeler = json.loads(p.read_text()).get("labeler_id", p.stem)
        for ep_id, lbl in labels.items():
            if ep_id in unified and unified[ep_id] != lbl:
                print(f"  CONFLICT episode {ep_id}: {unified[ep_id]} (kept) vs {lbl} ({labeler})")
                continue
            unified[ep_id] = lbl
    return unified


UNIFIED_LABELS = _merge_labels(APRIL9_LABEL_PATHS)
n_pos = sum(1 for v in UNIFIED_LABELS.values() if v == "contrail")
n_neg = sum(1 for v in UNIFIED_LABELS.values() if v == "no_contrail")
print(f"Unified labels: {len(UNIFIED_LABELS)} episodes "
      f"(contrail={n_pos}, no_contrail={n_neg})")

# --- Manifest + projections lookup ---
MANIFEST = json.loads(APRIL9_MANIFEST_PATH.read_text())
EPS_BY_ID: dict[int, dict] = {int(e["episode_id"]): e for e in MANIFEST["episodes"]}
VIDEO_T0 = datetime.datetime.fromisoformat(MANIFEST["video"]["start_utc"])
SECONDS_PER_FRAME = float(MANIFEST["video"]["seconds_per_frame"])

PROJ_IDX: dict[tuple[str, str], dict] = {}
with open(APRIL9_PROJECTIONS_PATH) as f:
    for line in f:
        row = json.loads(line)
        PROJ_IDX[(row["transponder_id"], row["wall_time_utc"])] = row
print(f"Loaded {len(PROJ_IDX)} projection rows")


def _peak_frame_wall(ep: dict) -> str | None:
    frames_list = ep.get("frames", [])
    if not frames_list:
        return None
    best = max(frames_list, key=lambda f: float(f.get("score") or 0.0))
    if float(best.get("score") or 0.0) > 0.0:
        return best["wall_time_utc"]
    return frames_list[len(frames_list) // 2]["wall_time_utc"]


# --- Build target list (one per labelled episode with a usable projection) ---
targets: list[dict] = []
n_no_manifest = n_no_proj = 0
for ep_id, label_str in UNIFIED_LABELS.items():
    ep = EPS_BY_ID.get(ep_id)
    if ep is None:
        n_no_manifest += 1
        continue
    wt = _peak_frame_wall(ep)
    if wt is None:
        continue
    proj = PROJ_IDX.get((ep["transponder_id"], wt))
    if proj is None:
        n_no_proj += 1
        continue
    dt = (datetime.datetime.fromisoformat(wt) - VIDEO_T0).total_seconds()
    fidx = int(round(dt / SECONDS_PER_FRAME))
    if fidx <= 0:
        continue
    targets.append({
        "episode_id": ep_id,
        "callsign": ep["callsign"],
        "label_raw": label_str,
        "frame_idx": fidx,
        "wall_time": wt,
        "pixel_x": float(proj["pixel_x"]),
        "pixel_y": float(proj["pixel_y"]),
        "path_dx": float(proj["path_dx"]),
        "path_dy": float(proj["path_dy"]),
        "roi": proj.get("roi", {"x": 0, "y": 0, "w": 0, "h": 0}),
    })
print(f"Built {len(targets)} targets "
      f"(skipped: no_manifest={n_no_manifest}, no_proj={n_no_proj})")

# --- Sequential decode: current frame + prev[1s] for every target ---
needed: set[int] = set()
for t in targets:
    needed.add(t["frame_idx"])
    if t["frame_idx"] - DEFAULT_LOOKBACK_S > 0:
        needed.add(t["frame_idx"] - DEFAULT_LOOKBACK_S)

print(f"Decoding {len(needed)} frames from {APRIL9_VIDEO_PATH} (single sequential pass)...")
frames = _decode_frames_sequential(APRIL9_VIDEO_PATH, needed)
print(f"  decoded {len(frames)} frames")

# --- Build records ---
april9_records: list[dict] = []
for t in targets:
    fidx = t["frame_idx"]
    frame = frames.get(fidx)
    if frame is None:
        continue
    crop, (tl_x, tl_y) = _context_crop(frame, t["pixel_x"], t["pixel_y"], CROP_PAD_PX)
    prev_by_L: dict[int, np.ndarray] = {}
    prev1 = frames.get(fidx - DEFAULT_LOOKBACK_S)
    if prev1 is not None:
        prev_crop, _ = _context_crop(prev1, t["pixel_x"], t["pixel_y"], CROP_PAD_PX)
        prev_by_L[DEFAULT_LOOKBACK_S] = prev_crop
    label = "positive" if t["label_raw"] == "contrail" else "negative"
    april9_records.append({
        "source": "april-9",
        "id": f"A9#{t['episode_id']:03d}",
        "label": label,
        "stratum": "all",
        "callsign": t["callsign"],
        "frame_idx": fidx,
        "pixel_x": t["pixel_x"],
        "pixel_y": t["pixel_y"],
        "path_dx": t["path_dx"],
        "path_dy": t["path_dy"],
        "roi": t["roi"],
        "crop": crop,
        "prev_crops_by_lookback": prev_by_L,
        "crop_tl": (tl_x, tl_y),
    })

pos = sum(1 for r in april9_records if r["label"] == "positive")
neg = sum(1 for r in april9_records if r["label"] == "negative")
print(f"April-9 records loaded: {len(april9_records)} (positive={pos}, negative={neg})")

  CONFLICT episode 9: no_contrail (kept) vs contrail (thendo)
Unified labels: 128 episodes (contrail=53, no_contrail=75)
Loaded 211004 projection rows
Built 99 targets (skipped: no_manifest=0, no_proj=0)
Decoding 198 frames from /net/d16/data/contrail-camera/2026_04_09_0000_2359.mp4 (single sequential pass)...
  decoded 198 frames
April-9 records loaded: 99 (positive=40, negative=59)


In [3]:
april8_records: list[dict] = []
if APRIL8_MANIFEST_PATH.exists() and APRIL8_LABELS_PATH.exists():
    april8_manifest = json.loads(APRIL8_MANIFEST_PATH.read_text())
    april8_labels = json.loads(APRIL8_LABELS_PATH.read_text())
    label_by_idx = {e["idx"]: e["label"] for e in april8_labels["labels"]}
    for c in april8_manifest["candidates"]:
        label = label_by_idx.get(c["idx"])
        if label != "positive":
            continue
        crop = None
        crop_tl: tuple[int, int] | None = None
        ctx_png_rel = c.get("context_png")
        if ctx_png_rel:
            ctx_path = APRIL8_DIR / ctx_png_rel
            if ctx_path.exists():
                ctx_img = cv2.imread(str(ctx_path))
                if ctx_img is not None:
                    crop = ctx_img
                    ctx_h, ctx_w = ctx_img.shape[:2]
                    crop_tl = (
                        max(0, int(c["pixel_x"]) - ctx_w // 2),
                        max(0, int(c["pixel_y"]) - ctx_h // 2),
                    )
        if crop is None:
            roi_png_path = APRIL8_DIR / c["roi_png"]
            crop = cv2.imread(str(roi_png_path))
            if crop is None:
                print(f"  SKIP April-8 idx {c['idx']}: failed to read {roi_png_path}")
                continue
            crop_tl = (
                max(0, int(c["roi"]["x"]) - EXTRACT_PAD),
                max(0, int(c["roi"]["y"]) - EXTRACT_PAD),
            )
        april8_records.append({
            "source": "april-8",
            "id": f"A8#{c['idx']:02d}",
            "label": "positive",
            "stratum": "sentinel",
            "callsign": c["callsign"],
            "frame_idx": int(c["frame_idx"]),
            "pixel_x": float(c["pixel_x"]),
            "pixel_y": float(c["pixel_y"]),
            "path_dx": float(c["path_dx"]),
            "path_dy": float(c["path_dy"]),
            "roi": c["roi"],
            "crop": crop,
            # April-8 PNGs are pre-rendered — no source video accessible from
            # the playground, so temporal_diff falls back to no-diff for these.
            "prev_crops_by_lookback": {},
            "crop_tl": crop_tl,
        })
    print(f"April-8 sentinels loaded: {len(april8_records)} positives")
else:
    print("April-8 sentinels skipped (manifest or labels missing)")

RECORDS: list[dict] = april9_records + april8_records
print(f"Total candidate records: {len(RECORDS)}")

April-8 sentinels loaded: 7 positives
Total candidate records: 106


In [4]:
def reconstruct_geometry(rec: dict, roi_along_px: int, roi_cross_px: int) -> tuple[Rect, np.ndarray, tuple[float, float]]:
    """Return (rect, polygon, path_vec) for a candidate record."""
    ch, cw = rec["crop"].shape[:2]
    tl_x, tl_y = rec["crop_tl"]
    center = PixelPoint(
        x=rec["pixel_x"] - tl_x,
        y=rec["pixel_y"] - tl_y,
    )
    path_vec = (rec["path_dx"], rec["path_dy"])
    dummy = DetectionConfig(
        roi_along_px=int(roi_along_px),
        roi_cross_px=int(roi_cross_px),
        roi_padding=20,
    )
    poly = rotated_polygon(center, path_vec, dummy)
    rect = Rect(x=0, y=0, w=cw, h=ch)
    return rect, poly, path_vec


def build_config(knobs: dict) -> DetectionConfig:
    return DetectionConfig(
        score_threshold=0.3,
        canny_low=int(knobs["canny_low"]),
        canny_high=int(knobs["canny_high"]),
        hough_threshold=int(knobs["hough_threshold"]),
        hough_min_line_length=int(knobs["hough_min_line_length"]),
        hough_max_line_gap=int(knobs["hough_max_line_gap"]),
        roi_padding=20,
        roi_along_px=int(knobs["roi_along_px"]),
        roi_cross_px=int(knobs["roi_cross_px"]),
        use_adaptive_canny=bool(knobs["use_adaptive_canny"]),
        canny_percentile_low=float(knobs["canny_percentile_low"]),
        canny_percentile_high=float(knobs["canny_percentile_high"]),
        canny_low_ratio=float(knobs["canny_low_ratio"]),
        canny_min_high=int(knobs["canny_min_high"]),
        angle_tolerance_deg=float(knobs["angle_tolerance_deg"]),
        long_line_min_px=float(knobs["long_line_min_px"]),
        score_fn="length",
        score_length_norm_px=float(knobs["score_length_norm_px"]),
        score_norm_count=6,
        use_rotated_mask=bool(knobs["use_rotated_mask"]),
        blur_kernel=int(knobs["blur_kernel"]),
        preprocessing="none",
    )


def _ensure_prev_for_lookback(rec: dict, lookback_s: int) -> np.ndarray | None:
    """Return the prev_crop for this record at the requested lookback,
    decoding it lazily on first access. Returns None if the record's source
    video isn't accessible (April-8 sentinels) or the prev frame is past
    the start of the video."""
    cache: dict[int, np.ndarray] = rec.setdefault("prev_crops_by_lookback", {})
    if lookback_s in cache:
        return cache[lookback_s]
    # April-8 sentinels have no source-video access from the playground.
    if rec["source"] != "april-9":
        return None
    fidx = rec["frame_idx"] - lookback_s
    if fidx < 0:
        return None
    prev_full = _decode_one_frame(APRIL9_VIDEO_PATH, fidx)
    if prev_full is None:
        return None
    prev_crop, _ = _context_crop(prev_full, rec["pixel_x"], rec["pixel_y"], CROP_PAD_PX)
    cache[lookback_s] = prev_crop
    return prev_crop


def run_chain_and_detect(rec: dict, chain: list[str], transform_params: dict, knobs: dict):
    """Apply the transform chain then run detect()."""
    rect, poly, path_vec = reconstruct_geometry(
        rec, int(knobs["roi_along_px"]), int(knobs["roi_cross_px"]),
    )
    lookback_s = int(knobs.get("prev_lookback_s", DEFAULT_LOOKBACK_S))
    prev_bgr = _ensure_prev_for_lookback(rec, lookback_s)
    gray = apply_chain(
        rec["crop"], chain,
        path_vec=path_vec,
        prev_bgr=prev_bgr,
        transform_params=transform_params,
    )
    cfg = build_config(knobs)
    result = detect(gray, rect, cfg, polygon=poly, path_vec=path_vec)
    return gray, result, rect, poly, path_vec, cfg

## Knob glossary

The sliders below drive **what preprocessing is applied before Canny** (the
transform chain) and **how Canny + Hough behave on the result**. Six knobs
worth memorising:

- **prev_lookback_s** + the `temporal_diff` chain element: the absolute
  difference between this frame and the one *N* seconds earlier (1 / 3 / 5
  / 10). Static structure (the building edge, terrain) cancels; new
  contrails remain. The live pipeline runs at 1 s today; longer lookbacks
  cancel jitter / sun-shadow drift better but ghost the aircraft itself.
- **pct_high / canny_percentile_high**: the Canny high threshold is set to
  this percentile of the masked-pixel distribution. Bigger → only the
  brightest edges survive. 99.0 catches fainter contrails; 99.8 is strict.
- **pct_low / canny_percentile_low**: the pixel-floor percentile.
  Everything dimmer than this is zeroed out *before* Canny so low-contrast
  sky texture can't trigger edges. Typical 96–98.
- **low_ratio / canny_low_ratio**: `canny_low = canny_low_ratio *
  canny_high`. 0.25 is wide hysteresis (picks up weak edges that connect
  to strong ones); 0.5–0.8 is narrow.
- **min_high / canny_min_high**: floor on `canny_high`. Prevents a
  low-contrast crop from collapsing thresholds to near-zero. Default 60.
- **score_length_norm_px**: the along-track pixel length at which the
  detector score saturates at 1.0. Production = 200 px.

In [5]:
# --- Candidate dropdown ---
def _label_tag(r):
    return r["label"] or "unlabeled"

_full = Layout(width="92%")
_half = Layout(width="46%")

cand_choices = [
    (f"{r['id']}  {r['callsign']:<10}  [{_label_tag(r)}]", i)
    for i, r in enumerate(RECORDS)
]
w_cand = Dropdown(options=cand_choices, description="candidate", layout=_full)

# --- Transform-chain dropdowns (A -> B -> C) ---
chain_options = ["none"] + [t[0] for t in TRANSFORMS]
w_ch1 = Dropdown(options=chain_options, value="none", description="chain A", layout=_half)
w_ch2 = Dropdown(options=chain_options, value="none", description="chain B", layout=_half)
w_ch3 = Dropdown(options=chain_options, value="none", description="chain C", layout=_half)

# --- Per-transform parameter sliders ---
w_lc_sigma = FloatSlider(value=25.0, min=5.0, max=60.0, step=1.0, description="lc_sigma", layout=_half)
w_dog_lo = FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="dog_lo", layout=_half)
w_dog_hi = FloatSlider(value=15.0, min=5.0, max=40.0, step=1.0, description="dog_hi", layout=_half)
w_frangi_lo = FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, description="frangi_σ_lo", layout=_half)
w_frangi_hi = FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description="frangi_σ_hi", layout=_half)
w_clahe_clip = FloatSlider(value=3.0, min=1.0, max=10.0, step=0.5, description="clahe_clip", layout=_half)
w_cross_gain = FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="cross_gain", layout=_half)
w_diff_gain = FloatSlider(value=4.0, min=1.0, max=20.0, step=0.5, description="diff_gain", layout=_half)

# --- Temporal-diff lookback (seconds back from current frame). Lazy-decoded. ---
w_prev_lookback = Dropdown(
    options=[(f"{s} s", s) for s in PREV_LOOKBACKS_S],
    value=DEFAULT_LOOKBACK_S, description="prev_lookback", layout=_half,
)

# --- ROI geometry sliders ---
w_roi_along = IntSlider(value=int(DET_CFG.roi_along_px), min=60, max=600, step=10, description="roi_along", layout=_half)
w_roi_cross = IntSlider(value=int(DET_CFG.roi_cross_px), min=20, max=240, step=5, description="roi_cross", layout=_half)

# --- Detector knobs ---
w_use_adapt = Checkbox(value=bool(DET_CFG.use_adaptive_canny), description="adaptive Canny")
w_use_mask = Checkbox(value=bool(DET_CFG.use_rotated_mask), description="rotated mask")

w_pct_high = FloatSlider(value=float(DET_CFG.canny_percentile_high), min=95.0, max=99.9, step=0.1, description="pct_high", layout=_half)
w_pct_low = FloatSlider(value=float(DET_CFG.canny_percentile_low), min=80.0, max=99.0, step=0.5, description="pct_low", layout=_half)
w_low_ratio = FloatSlider(value=float(DET_CFG.canny_low_ratio), min=0.1, max=0.8, step=0.05, description="low_ratio", layout=_half)
w_min_high = IntSlider(value=int(DET_CFG.canny_min_high), min=10, max=200, step=5, description="min_high", layout=_half)

w_canny_low = IntSlider(value=int(DET_CFG.canny_low), min=1, max=255, step=1, description="canny_low", layout=_half)
w_canny_high = IntSlider(value=int(DET_CFG.canny_high), min=1, max=400, step=1, description="canny_high", layout=_half)

w_blur = IntSlider(value=int(DET_CFG.blur_kernel), min=0, max=11, step=1, description="blur", layout=_half)

w_hough_thr = IntSlider(value=int(DET_CFG.hough_threshold), min=5, max=100, step=1, description="hough_thr", layout=_half)
w_hough_minL = IntSlider(value=int(DET_CFG.hough_min_line_length), min=5, max=80, step=1, description="hough_minL", layout=_half)
w_hough_gap = IntSlider(value=int(DET_CFG.hough_max_line_gap), min=1, max=30, step=1, description="hough_gap", layout=_half)

w_tol = FloatSlider(value=float(DET_CFG.angle_tolerance_deg), min=1.0, max=45.0, step=0.5, description="angle_tol", layout=_half)
w_longL = FloatSlider(value=float(DET_CFG.long_line_min_px), min=5.0, max=80.0, step=1.0, description="long_min", layout=_half)
w_score_norm = FloatSlider(value=float(DET_CFG.score_length_norm_px), min=30.0, max=400.0, step=10.0, description="score_norm_px", layout=_half)

controls = VBox([
    w_cand,
    Label(value="Transform chain (applied left-to-right before Canny):"),
    HBox([w_ch1, w_ch2, w_ch3]),
    Label(value="Per-transform parameters:"),
    HBox([w_lc_sigma, w_dog_lo]),
    HBox([w_dog_hi, w_frangi_lo]),
    HBox([w_frangi_hi, w_clahe_clip]),
    HBox([w_cross_gain, w_diff_gain]),
    Label(value="Temporal-diff lookback (only used when chain contains temporal_diff):"),
    HBox([w_prev_lookback]),
    Label(value="ROI polygon geometry:"),
    HBox([w_roi_along, w_roi_cross]),
    Label(value="Canny + Hough knobs:"),
    HBox([w_use_adapt, w_use_mask]),
    HBox([w_pct_high, w_pct_low]),
    HBox([w_low_ratio, w_min_high]),
    HBox([w_canny_low, w_canny_high]),
    HBox([w_blur, w_hough_thr]),
    HBox([w_hough_minL, w_hough_gap]),
    HBox([w_tol, w_longL]),
    HBox([w_score_norm]),
])

def _current_chain() -> list[str]:
    return [w_ch1.value, w_ch2.value, w_ch3.value]

def _current_transform_params() -> dict:
    return {
        "local_contrast": {"local_contrast_sigma": float(w_lc_sigma.value)},
        "dog": {"dog_sigma_low": float(w_dog_lo.value), "dog_sigma_high": float(w_dog_hi.value)},
        "frangi": {
            "frangi_sigma_min": float(w_frangi_lo.value),
            "frangi_sigma_max": float(w_frangi_hi.value),
        },
        "clahe": {"clahe_clip_limit": float(w_clahe_clip.value)},
        "cross_grad": {"cross_grad_gain": float(w_cross_gain.value)},
        "temporal_diff": {"temporal_diff_gain": float(w_diff_gain.value)},
    }

def _current_knobs() -> dict:
    return {
        "use_adaptive_canny": w_use_adapt.value,
        "use_rotated_mask": w_use_mask.value,
        "canny_percentile_high": w_pct_high.value,
        "canny_percentile_low": w_pct_low.value,
        "canny_low_ratio": w_low_ratio.value,
        "canny_min_high": w_min_high.value,
        "canny_low": w_canny_low.value,
        "canny_high": w_canny_high.value,
        "blur_kernel": w_blur.value,
        "hough_threshold": w_hough_thr.value,
        "hough_min_line_length": w_hough_minL.value,
        "hough_max_line_gap": w_hough_gap.value,
        "angle_tolerance_deg": w_tol.value,
        "long_line_min_px": w_longL.value,
        "score_length_norm_px": w_score_norm.value,
        "roi_along_px": w_roi_along.value,
        "roi_cross_px": w_roi_cross.value,
        "prev_lookback_s": int(w_prev_lookback.value),
    }

In [6]:
render_out = Output()


def _angle_delta(a: float, b: float) -> float:
    return abs(((a - b + 90.0) % 180.0) - 90.0)


def _render_panels(**kwargs):
    idx = kwargs["cand_idx"]
    rec = RECORDS[idx]
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()

    gray, result, rect, poly, path_vec, cfg = run_chain_and_detect(rec, chain, params, knobs)

    mask = None
    base = gray
    if cfg.blur_kernel and cfg.blur_kernel > 1:
        k = int(cfg.blur_kernel) | 1
        base = cv2.GaussianBlur(base, (k, k), 0)
    if cfg.use_rotated_mask:
        mask = np.zeros(base.shape, dtype=np.uint8)
        cv2.fillPoly(mask, [poly.astype(np.int32)], 255)
        masked_values = base[mask > 0]
    else:
        masked_values = base.reshape(-1)
    if cfg.use_adaptive_canny and masked_values.size:
        p_hi = float(np.percentile(masked_values, cfg.canny_percentile_high))
        p_lo = float(np.percentile(masked_values, cfg.canny_percentile_low))
        canny_high = max(int(round(p_hi)), int(cfg.canny_min_high))
        canny_low = max(1, int(round(canny_high * cfg.canny_low_ratio)))
        floor = int(round(p_lo))
    else:
        canny_high, canny_low, floor = int(cfg.canny_high), int(cfg.canny_low), 0
    crop_for_canny = base.copy()
    if mask is not None:
        crop_for_canny = cv2.bitwise_and(crop_for_canny, crop_for_canny, mask=mask)
    if floor > 0:
        _, crop_for_canny = cv2.threshold(crop_for_canny, floor, 255, cv2.THRESH_TOZERO)
    edges = cv2.Canny(crop_for_canny, canny_low, canny_high)
    if mask is not None:
        edges = cv2.bitwise_and(edges, edges, mask=mask)
    raw = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi / 180.0,
        threshold=int(cfg.hough_threshold),
        minLineLength=int(cfg.hough_min_line_length),
        maxLineGap=int(cfg.hough_max_line_gap),
    )
    raw_lines = [] if raw is None else [tuple(int(v) for v in ln[0]) for ln in raw]

    path_angle = math.degrees(math.atan2(path_vec[1], path_vec[0])) % 180.0
    tol = float(cfg.angle_tolerance_deg)

    def _aligned(x1, y1, x2, y2):
        a = math.degrees(math.atan2(y2 - y1, x2 - x1)) % 180.0
        return _angle_delta(a, path_angle) <= tol

    with render_out:
        render_out.clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        crop_rgb = cv2.cvtColor(rec["crop"], cv2.COLOR_BGR2RGB)
        tl_x, tl_y = rec["crop_tl"]
        cx = rec["pixel_x"] - tl_x
        cy = rec["pixel_y"] - tl_y

        axes[0, 0].imshow(crop_rgb)
        poly_closed = np.vstack([poly, poly[:1]])
        axes[0, 0].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.5)
        L = 60.0
        axes[0, 0].plot(
            [cx, cx - L * path_vec[0]],
            [cy, cy - L * path_vec[1]],
            color="#ff6030", lw=1.4, alpha=0.9,
        )
        axes[0, 0].scatter([cx], [cy], c="#ff6030", s=24, zorder=5)
        axes[0, 0].set_title(
            f"{rec['id']} {rec['callsign']}  [{_label_tag(rec)}]  "
            f"crop {crop_rgb.shape[1]}×{crop_rgb.shape[0]}  path={path_angle:5.1f}°"
        )
        axes[0, 0].axis("off")

        last_tf = next((c for c in reversed(chain) if c and c != "none"), None)
        cmap = DISPLAY_COLORMAPS.get(last_tf, "gray") if last_tf else "gray"
        axes[0, 1].imshow(gray, cmap=cmap)
        chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"
        lookback_note = ""
        if "temporal_diff" in chain:
            lookback_note = f"  prev={knobs['prev_lookback_s']}s"
        axes[0, 1].set_title(f"chain output: {chain_str}{lookback_note}")
        axes[0, 1].axis("off")

        axes[1, 0].imshow(edges, cmap="gray")
        axes[1, 0].set_title(
            f"Canny edges  canny_low={canny_low} canny_high={canny_high} floor={floor}"
        )
        axes[1, 0].axis("off")

        axes[1, 1].imshow(crop_rgb)
        axes[1, 1].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.0)
        n_aligned = n_rejected = 0
        for (x1, y1, x2, y2) in raw_lines:
            if _aligned(x1, y1, x2, y2):
                axes[1, 1].plot([x1, x2], [y1, y2], color="#50e050", lw=1.0, alpha=0.9)
                n_aligned += 1
            else:
                axes[1, 1].plot([x1, x2], [y1, y2], color="#ff9a40", lw=0.8, alpha=0.55)
                n_rejected += 1
        if result.pixel_line is not None:
            x1, y1, x2, y2 = result.pixel_line
            axes[1, 1].plot([x1, x2], [y1, y2], color="#30ff30", lw=2.8)
        axes[1, 1].set_title(
            f"score={result.score:.3f}  long={result.num_long_lines}  aligned={n_aligned}  "
            f"rej={n_rejected}  len_px={result.contrail_length_px:.0f}"
        )
        axes[1, 1].axis("off")
        plt.tight_layout()
        plt.show()


_all_widgets = [
    w_cand, w_ch1, w_ch2, w_ch3,
    w_lc_sigma, w_dog_lo, w_dog_hi, w_frangi_lo, w_frangi_hi, w_clahe_clip,
    w_cross_gain, w_diff_gain, w_prev_lookback,
    w_roi_along, w_roi_cross,
    w_use_adapt, w_use_mask, w_pct_high, w_pct_low, w_low_ratio, w_min_high,
    w_canny_low, w_canny_high, w_blur, w_hough_thr, w_hough_minL, w_hough_gap,
    w_tol, w_longL, w_score_norm,
]
ui_out = interactive_output(_render_panels, {"cand_idx": w_cand})
for w in _all_widgets:
    if w is not w_cand:
        w.observe(lambda *_: _render_panels(cand_idx=w_cand.value), names="value")

display(controls, render_out)

Output()

## Add user-suggested candidates

The 35-candidate stratified batch can't be the whole picture — you've seen
contrails in the April-9 video on other flights that didn't land in the
batch. This cell lets you add them as labelled records in `RECORDS` so the
matrix / Bulk AUC / live-render pick them up.

Format of `USER_CANDIDATES` (edit the list below):
- `(callsign, time_iso_utc, label, note)`
- `time_iso_utc` is the UTC peak moment for the contrail (e.g.
  `"2026-04-09T13:14:00+00:00"`). If you have an ET time, convert it: ET on
  April 9 is UTC-4 (DST), so 9:50 ET = 13:50 UTC.
- `label` is one of `"positive"` (contrail) / `"negative"` (no contrail) /
  `"unsure"`.
- `note` is free-form (optional), surfaced in the dropdown and matrix title.

After editing, run the cell to re-decode the relevant video frames, append
records with source=`"user"` and stratum=`"user_added"`, and rebuild the
widget dropdown so the new rows appear in the candidate selector.

In [ ]:
from datetime import datetime, timezone, timedelta
from concam.adsb import Ping
from concam.projection import Calibration, PixelPoint, load_calibration, project_pings

# Edit this list; re-run the cell to refresh RECORDS.
USER_CANDIDATES: list[tuple[str, str, str, str]] = [
    # (callsign, time_iso_utc, label, note)
    ("MNB312", "2026-04-09T13:14:19+00:00", "positive", "seen in video"),
    ("CHG591", "2026-04-09T13:51:16+00:00", "positive", "seen in video"),
    ("AFR2N",  "2026-04-09T13:54:47+00:00", "positive", "seen in video"),
    ("VIR91U", "2026-04-09T14:08:59+00:00", "positive", "seen in video"),
]

OCR_PATH = REPO_ROOT / "output" / APRIL9_DATE / "ocr.jsonl"
with open(OCR_PATH) as f:
    first_line = json.loads(f.readline())
USER_VIDEO_T0 = datetime.fromisoformat(first_line["wall_time_utc"])
print(f"April-9 video starts at frame 0 = {USER_VIDEO_T0.isoformat()}")

ADSB_PATH = REPO_ROOT / "output" / APRIL9_DATE / "adsb.json"
ADSB = json.loads(ADSB_PATH.read_text())

CALIB = load_calibration(SITE_CONFIG.calibration)


def _ping_from_dict(d: dict) -> Ping:
    return Ping(
        time=datetime.fromisoformat(d["time"]),
        lat=float(d["lat"]),
        lon=float(d["lon"]),
        alt_m=float(d["alt_m"]),
        alt_gnss_m=d.get("alt_gnss_m"),
        alt_baro_m=d.get("alt_baro_m"),
        alt_source=d.get("alt_source", "gnss"),
    )


def _resolve_user_candidate(callsign: str, t_iso: str, label: str, note: str):
    t_target = datetime.fromisoformat(t_iso).astimezone(timezone.utc)
    best = None
    best_dt = None
    for flight in ADSB:
        if (flight.get("callsign") or "").strip().upper() != callsign.upper():
            continue
        for i, p in enumerate(flight["pings"]):
            pt = datetime.fromisoformat(p["time"]).astimezone(timezone.utc)
            dt = abs((pt - t_target).total_seconds())
            if best_dt is None or dt < best_dt:
                best_dt = dt
                best = (flight, i, p, pt)
    if best is None:
        print(f"  WARN: {callsign} not found in adsb.json; skipping")
        return None
    flight, i_ping, ping, ping_t = best
    if best_dt > 60.0:
        print(f"  WARN: {callsign} nearest ping is {best_dt:.0f}s from target; skipping")
        return None

    proj = project_pings([_ping_from_dict(ping)], CALIB)[0]
    if proj is None:
        print(f"  WARN: {callsign} at {ping_t.strftime('%H:%M:%S')}Z projects outside frame; skipping")
        return None

    pings_list = flight["pings"]
    i_back = max(0, i_ping - 5)
    i_fwd = min(len(pings_list) - 1, i_ping + 5)
    nbrs = project_pings(
        [_ping_from_dict(pings_list[i_back]), _ping_from_dict(pings_list[i_fwd])],
        CALIB,
    )
    if nbrs[0] is None or nbrs[1] is None:
        path_dx, path_dy = 1.0, 0.0
    else:
        dx = nbrs[1].x - nbrs[0].x
        dy = nbrs[1].y - nbrs[0].y
        mag = (dx * dx + dy * dy) ** 0.5
        path_dx, path_dy = (dx / mag, dy / mag) if mag > 1e-3 else (1.0, 0.0)

    frame_idx = int(round((ping_t - USER_VIDEO_T0).total_seconds()))
    if frame_idx < 0:
        print(f"  WARN: {callsign} frame_idx={frame_idx} is negative; skipping")
        return None

    return {
        "callsign": callsign,
        "transponder_id": flight["transponder_id"],
        "pixel_x": proj.x,
        "pixel_y": proj.y,
        "path_dx": path_dx,
        "path_dy": path_dy,
        "frame_idx": frame_idx,
        "ping_time": ping_t,
        "label": label,
        "note": note,
    }


user_resolved = [_resolve_user_candidate(*c) for c in USER_CANDIDATES]
user_resolved = [r for r in user_resolved if r is not None]
user_needed: set[int] = set()
for r in user_resolved:
    user_needed.add(r["frame_idx"])
    if r["frame_idx"] - DEFAULT_LOOKBACK_S > 0:
        user_needed.add(r["frame_idx"] - DEFAULT_LOOKBACK_S)

already = set(frames.keys())
missing = user_needed - already
if missing:
    print(f"Decoding {len(missing)} additional frames for user candidates ...")
    new_frames = _decode_frames_sequential(APRIL9_VIDEO_PATH, missing)
    frames.update(new_frames)

# Idempotent: drop user records from a previous run.
RECORDS[:] = [r for r in RECORDS if r["source"] != "user"]

for r in user_resolved:
    fidx = r["frame_idx"]
    frame = frames.get(fidx)
    if frame is None:
        print(f"  SKIP {r['callsign']}: frame {fidx} missing")
        continue
    crop, (tl_x, tl_y) = _context_crop(frame, r["pixel_x"], r["pixel_y"], CROP_PAD_PX)
    prev_by_L: dict[int, np.ndarray] = {}
    prev1 = frames.get(fidx - DEFAULT_LOOKBACK_S)
    if prev1 is not None:
        prev_crop, _ = _context_crop(prev1, r["pixel_x"], r["pixel_y"], CROP_PAD_PX)
        prev_by_L[DEFAULT_LOOKBACK_S] = prev_crop
    RECORDS.append({
        "source": "user",
        "id": f"U#{r['callsign']}",
        "label": r["label"],
        "stratum": "user_added",
        "callsign": r["callsign"],
        "frame_idx": fidx,
        "pixel_x": r["pixel_x"],
        "pixel_y": r["pixel_y"],
        "path_dx": r["path_dx"],
        "path_dy": r["path_dy"],
        "roi": {"x": int(r["pixel_x"]) - 90, "y": int(r["pixel_y"]) - 20, "w": 180, "h": 40},
        "crop": crop,
        "prev_crops_by_lookback": prev_by_L,
        "crop_tl": (tl_x, tl_y),
        "note": r["note"],
    })
    print(
        f"  + {r['callsign']} @ {r['ping_time'].strftime('%H:%M:%S')}Z  "
        f"pixel=({r['pixel_x']:.0f}, {r['pixel_y']:.0f})  label={r['label']}"
    )

w_cand.options = [
    (f"{r['id']}  {r['callsign']:<10}  [{_label_tag(r)}]", i)
    for i, r in enumerate(RECORDS)
]

a9_user_pos = sum(1 for r in RECORDS if r["source"] == "user" and r["label"] == "positive")
a9_user_neg = sum(1 for r in RECORDS if r["source"] == "user" and r["label"] == "negative")
print(
    f"\nUser candidates merged. Total RECORDS: {len(RECORDS)} "
    f"(user positive={a9_user_pos}, user negative={a9_user_neg})"
)

## Confusion-matrix view (TP / FN / FP / TN)

Render a coloured grid showing every labelled record at the current chain
+ knob settings, partitioned by confusion-matrix bucket:

- **green** = TP (positive label, score ≥ Youden-J threshold)
- **red** = FN (positive label, below threshold)
- **orange** = FP (negative label, above threshold)
- **blue** = TN (negative label, below threshold)

The Youden-J threshold is computed from the April-9 positives vs negatives
each render (matches the live-render pane). Use the **filter** dropdown to
focus on one bucket — useful when the full grid is too big to scan.

This is where the temporal-diff lookback experiment becomes concrete: set
the chain to `temporal_diff → cross_grad`, switch the **prev_lookback**
knob between 1 / 3 / 5 / 10, and watch which FPs fall out and which TPs
get hurt.

In [7]:
matrix_grid_out = Output()


w_matrix_filter = Dropdown(
    options=[
        ("all (TP+FN+FP+TN)", "all"),
        ("TP only", "TP"),
        ("FN only", "FN"),
        ("FP only", "FP"),
        ("TN only", "TN"),
    ],
    value="all",
    description="filter",
    layout=Layout(width="40%"),
)


def render_confusion_matrix(_button=None):
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()
    show = w_matrix_filter.value

    if not RECORDS:
        with matrix_grid_out:
            matrix_grid_out.clear_output()
            print("No records loaded.")
        return

    results_by_id: dict[str, dict] = {}
    for rec in RECORDS:
        gray, result, rect, poly, pv, cfg = run_chain_and_detect(rec, chain, params, knobs)
        results_by_id[rec["id"]] = {
            "rec": rec, "gray": gray, "result": result,
            "poly": poly, "path_vec": pv, "cfg": cfg,
        }

    a9_pos = [
        results_by_id[r["id"]]["result"].score for r in RECORDS
        if r["source"] == "april-9" and r["label"] == "positive"
    ]
    a9_neg = [
        results_by_id[r["id"]]["result"].score for r in RECORDS
        if r["source"] == "april-9" and r["label"] == "negative"
    ]
    if a9_pos and a9_neg:
        scores = sorted(set(a9_pos + a9_neg))
        cands = [(a + b) / 2 for a, b in zip(scores[:-1], scores[1:])] or [0.0]
        best_t, best_j = cands[0], -1.0
        for t in cands:
            tpr = sum(1 for p in a9_pos if p >= t) / len(a9_pos)
            fpr = sum(1 for n in a9_neg if n >= t) / len(a9_neg)
            if tpr - fpr > best_j:
                best_j, best_t = tpr - fpr, t
        threshold = float(best_t)
    else:
        threshold = float(SITE_CONFIG.aggregation.detection_threshold)

    bucket_records: list[tuple[dict, str, str]] = []
    for rec in RECORDS:
        if rec["label"] not in ("positive", "negative"):
            continue
        info = results_by_id[rec["id"]]
        score = float(info["result"].score)
        is_pos = (rec["label"] == "positive")
        is_above = score >= threshold
        if is_pos and is_above:
            bucket, color = "TP", "#30c030"
        elif is_pos and not is_above:
            bucket, color = "FN", "#e03030"
        elif (not is_pos) and is_above:
            bucket, color = "FP", "#ff8800"
        else:
            bucket, color = "TN", "#3060e0"
        if show == "all" or show == bucket:
            bucket_records.append((rec, bucket, color))

    counts = {"TP": 0, "FN": 0, "FP": 0, "TN": 0}
    for rec in RECORDS:
        if rec["label"] not in ("positive", "negative"):
            continue
        info = results_by_id[rec["id"]]
        score = float(info["result"].score)
        is_pos = (rec["label"] == "positive")
        is_above = score >= threshold
        if is_pos and is_above:
            counts["TP"] += 1
        elif is_pos and not is_above:
            counts["FN"] += 1
        elif (not is_pos) and is_above:
            counts["FP"] += 1
        else:
            counts["TN"] += 1

    if not bucket_records:
        with matrix_grid_out:
            matrix_grid_out.clear_output(wait=True)
            print(f"No records in filter '{show}'.  Counts: {counts}  threshold={threshold:.3f}")
        return

    # Sort within bucket: bring most-extreme cases first (FPs by descending
    # score, FNs by ascending score, TPs by ascending score, TNs by
    # descending score) so the user sees the worst offenders first.
    def _sort_key(item):
        rec, bucket, _ = item
        score = float(results_by_id[rec["id"]]["result"].score)
        if bucket == "FP":
            return (0, -score)
        if bucket == "FN":
            return (1, score)
        if bucket == "TP":
            return (2, score)
        return (3, -score)
    bucket_records.sort(key=_sort_key)

    n = len(bucket_records)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.0, nrows * 3.6))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes.reshape(1, -1)
    elif ncols == 1:
        axes = axes.reshape(-1, 1)

    for idx, (rec, bucket, color) in enumerate(bucket_records):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        info = results_by_id[rec["id"]]
        score = float(info["result"].score)

        crop_rgb = cv2.cvtColor(rec["crop"], cv2.COLOR_BGR2RGB)
        ax.imshow(crop_rgb)

        poly = info["poly"]
        poly_closed = np.vstack([poly, poly[:1]])
        ax.plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.0)

        tl_x, tl_y = rec["crop_tl"]
        cx = rec["pixel_x"] - tl_x
        cy = rec["pixel_y"] - tl_y
        L = 60.0
        ax.plot(
            [cx, cx - L * info["path_vec"][0]],
            [cy, cy - L * info["path_vec"][1]],
            color="#ff6030", lw=1.0, alpha=0.9,
        )
        ax.scatter([cx], [cy], c="#ff6030", s=14, zorder=5)

        if info["result"].pixel_line is not None:
            x1, y1, x2, y2 = info["result"].pixel_line
            ax.plot([x1, x2], [y1, y2], color="#30ff30", lw=2.0)

        ax.set_title(
            f"{rec['id']}  {rec['callsign']}\n{bucket}   score={score:.3f}",
            color=color, fontsize=9,
        )
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    for idx in range(len(bucket_records), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].axis("off")

    chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"
    lookback_note = ""
    if "temporal_diff" in chain:
        lookback_note = f"   prev={knobs['prev_lookback_s']}s"
    fig.suptitle(
        f"Confusion matrix [{show}]   chain: {chain_str}{lookback_note}\n"
        f"threshold={threshold:.3f}   "
        f"TP={counts['TP']}  FN={counts['FN']}  FP={counts['FP']}  TN={counts['TN']}",
        y=1.0, fontsize=11,
    )

    with matrix_grid_out:
        matrix_grid_out.clear_output(wait=True)
        plt.tight_layout()
        plt.show()


matrix_button = Button(
    description="Render confusion matrix",
    button_style="info",
    layout=Layout(width="280px"),
)
matrix_button.on_click(render_confusion_matrix)
display(HBox([matrix_button, w_matrix_filter]), matrix_grid_out)

Output()

## Bulk evaluation

Press **Run Bulk AUC** to score the entire candidate pool at the current
chain + knobs. Reports:

- **April-9 AUC** (Mann-Whitney) on the unified-label positives vs negatives.
- **Youden-J threshold** and the **recall / FP count** at that threshold.
- **April-8 sentinel pass rate** at the same threshold (regression check).
- Histogram of positive vs negative scores with the threshold line overlaid.

Useful for "did this knob change actually move AUC, or just shuffle a few
borderline cases?" — the histograms make the answer visual.

In [8]:
bulk_out = Output()


def _mann_whitney_auc(pos: list[float], neg: list[float]) -> float:
    if not pos or not neg:
        return 0.5
    wins = 0.0
    for p in pos:
        for n in neg:
            if p > n:
                wins += 1.0
            elif p == n:
                wins += 0.5
    return wins / (len(pos) * len(neg))


def _best_threshold(pos: list[float], neg: list[float]) -> tuple[float, float]:
    if not pos or not neg:
        return 0.5, 0.0
    scores = sorted(set(pos + neg))
    if len(scores) == 1:
        return scores[0] - 1e-6, 0.0
    cands = [(a + b) / 2 for a, b in zip(scores[:-1], scores[1:])]
    best_t, best_j = cands[0], -1.0
    for t in cands:
        tpr = sum(1 for p in pos if p >= t) / len(pos)
        fpr = sum(1 for n in neg if n >= t) / len(neg)
        if tpr - fpr > best_j:
            best_j, best_t = tpr - fpr, t
    return best_t, best_j


def run_bulk_eval(_button=None):
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()

    all_scores: dict[str, float] = {}
    for rec in RECORDS:
        _, result, *_ = run_chain_and_detect(rec, chain, params, knobs)
        all_scores[rec["id"]] = float(result.score)

    a9_pos = [all_scores[r["id"]] for r in RECORDS
              if r["source"] == "april-9" and r["label"] == "positive"]
    a9_neg = [all_scores[r["id"]] for r in RECORDS
              if r["source"] == "april-9" and r["label"] == "negative"]
    a8_pos = [all_scores[r["id"]] for r in RECORDS if r["source"] == "april-8"]

    auc = _mann_whitney_auc(a9_pos, a9_neg)
    threshold, youden_j = _best_threshold(a9_pos, a9_neg)

    recall = sum(1 for p in a9_pos if p >= threshold)
    fp_count = sum(1 for n in a9_neg if n >= threshold)
    a8_pass = sum(1 for p in a8_pos if p >= threshold)

    chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"
    lookback_note = f"  prev_lookback={knobs['prev_lookback_s']}s" if "temporal_diff" in chain else ""

    with bulk_out:
        bulk_out.clear_output(wait=True)
        print(f"=== Bulk evaluation — chain: {chain_str}{lookback_note} ===")
        print()
        print(f"April-9 labelled: {len(a9_pos)} positives, {len(a9_neg)} negatives")
        print(f"April-8 sentinels: {len(a8_pos)} positives")
        print()
        print(f"AUC (Mann-Whitney):      {auc:.3f}")
        print(f"Youden-J threshold:      {threshold:.3f}  (J = {youden_j:.3f})")
        print(f"April-9 recall @ t:      {recall}/{len(a9_pos)}")
        print(f"April-9 FP @ t:          {fp_count}/{len(a9_neg)}")
        print(f"April-8 sentinel @ t:    {a8_pass}/{len(a8_pos)}   (regression check)")

        fig, ax = plt.subplots(figsize=(10, 3.5))
        bins = np.linspace(0, 1, 21)
        if a9_neg:
            ax.hist(a9_neg, bins=bins, alpha=0.6, label=f"A9 negative (n={len(a9_neg)})", color="#3c78d8")
        if a9_pos:
            ax.hist(a9_pos, bins=bins, alpha=0.6, label=f"A9 positive (n={len(a9_pos)})", color="#e06666")
        if a8_pos:
            ax.hist(a8_pos, bins=bins, alpha=0.5, label=f"A8 sentinel (n={len(a8_pos)})", color="#f1c232")
        ax.axvline(threshold, color="k", ls="--", label=f"Youden-J thr={threshold:.3f}")
        ax.set_title(
            f"AUC={auc:.3f}   recall={recall}/{len(a9_pos)}   FP={fp_count}/{len(a9_neg)}{lookback_note}"
        )
        ax.set_xlabel("detector score")
        ax.set_ylabel("count")
        ax.legend(loc="upper right", fontsize=9)
        plt.tight_layout()
        plt.show()


run_button = Button(description="Run Bulk AUC", button_style="primary", layout=Layout(width="200px"))
run_button.on_click(run_bulk_eval)
display(run_button, bulk_out)

Button(button_style='primary', description='Run Bulk AUC', layout=Layout(width='200px'), style=ButtonStyle())

Output()

## Export current settings

Once you've found a chain + knob configuration you like, run the cell below
to print a YAML block you can paste into `configs/mit_green_building.yaml`.

Note: wiring `transform_chain` into `concam.detection.detect()` so the live
pipeline honours it is an explicit follow-up PRD item — this notebook
produces the YAML snippet but you'll need that follow-up item to actually
apply it to full-day runs.

In [ ]:
def export_yaml():
    chain = [c for c in _current_chain() if c and c != "none"]
    params = _current_transform_params()
    knobs = _current_knobs()

    def _fmt(v):
        if isinstance(v, bool):
            return str(v).lower()
        if isinstance(v, float):
            return f"{v:g}"
        return str(v)

    out = ["detection:"]
    out.append("  # Chain applied left-to-right before Canny:")
    out.append(f"  transform_chain: [{', '.join(chain) if chain else '# empty — passthrough'}]")
    active_params = {k: v for k, v in params.items() if k in chain}
    if "temporal_diff" in chain:
        active_params.setdefault("temporal_diff", {})["prev_lookback_s"] = knobs["prev_lookback_s"]
    if active_params:
        out.append("  transform_params:")
        for tf_name, pkvs in active_params.items():
            out.append(f"    {tf_name}:")
            for pk, pv in pkvs.items():
                out.append(f"      {pk}: {_fmt(pv)}")
    out.append(f"  use_adaptive_canny: {_fmt(knobs['use_adaptive_canny'])}")
    out.append(f"  use_rotated_mask: {_fmt(knobs['use_rotated_mask'])}")
    out.append(f"  canny_percentile_high: {_fmt(knobs['canny_percentile_high'])}")
    out.append(f"  canny_percentile_low: {_fmt(knobs['canny_percentile_low'])}")
    out.append(f"  canny_low_ratio: {_fmt(knobs['canny_low_ratio'])}")
    out.append(f"  canny_min_high: {_fmt(knobs['canny_min_high'])}")
    out.append(f"  canny_low: {_fmt(knobs['canny_low'])}")
    out.append(f"  canny_high: {_fmt(knobs['canny_high'])}")
    out.append(f"  blur_kernel: {_fmt(knobs['blur_kernel'])}")
    out.append(f"  hough_threshold: {_fmt(knobs['hough_threshold'])}")
    out.append(f"  hough_min_line_length: {_fmt(knobs['hough_min_line_length'])}")
    out.append(f"  hough_max_line_gap: {_fmt(knobs['hough_max_line_gap'])}")
    out.append(f"  angle_tolerance_deg: {_fmt(knobs['angle_tolerance_deg'])}")
    out.append(f"  long_line_min_px: {_fmt(knobs['long_line_min_px'])}")
    out.append(f"  score_length_norm_px: {_fmt(knobs['score_length_norm_px'])}")
    out.append(f"  roi_along_px: {_fmt(knobs['roi_along_px'])}")
    out.append(f"  roi_cross_px: {_fmt(knobs['roi_cross_px'])}")
    print("\n".join(out))


export_yaml()